<div style="background: #86d1f1ff; border-radius: 5px; padding: 1rem; margin-bottom: 1rem">
<img src="https://store.utec.edu.pe/files/Recursos/logo-utec-h.png" alt="Banner" width="150" />   
<div style="font-weight: bold; color: #434549ff; float: right "><u style="font-size: 28px;">Base de Datos II</u> <br />
<span style="float:right"> Profesor Heider Sanchez</span> <br /> 
<span style="float:right">  2026 - 1 </span>   
</div> </div>

# Laboratorio 8.1: Procesamiento de Textos y Bag of Words

> **Prof. Heider Sanchez**  

##  Introducción
Este laboratorio tiene como objetivo el análisis y búsqueda de documentos textuales utilizando procesamiento de lenguaje natural (NLP) y una base de datos PostgreSQL. Se trabajará paso a paso desde la extracción de los textos hasta la aplicación búsquedas booleanas.


### Objetivos
- Configurar la tabla en PostgreSQL y carga de datos.
- Desde Python leer los textos desde PostgreSQL.
- Realizar el procesamiento de textos: convertir a minúscula, tokenización, stopwords, stemming y frecuencia de términos.
- Almacenar los Bag of Words en la base de datos en formato JSON.
- Realizar búsquedas de documentos similares a una consulta booleana (conectores AND, OR y AND-NOT).


### Requisitos previos

- Tener instalado PostgreSQL en su computadora (ultima versión)
- Tener instalado las siguientes dependencias en Python:

    ```bash
    pip install psycopg2-binary nltk scikit-learn pandas
    ```

- Opcionalmente descargar los recursos de NLTK:

    ```python
    import nltk
    nltk.download('punkt')
    ```


## 1. (2 puntos) Configurar la tabla en PostgreSQL y carga de datos


### Crear las tablas

Crear la tabla en PostgreSQL para almacenar los textos de noticias y el bag of words:

```sql
CREATE TABLE noticias (
    id SERIAL PRIMARY KEY,
    url TEXT,
    contenido TEXT,
    categoria VARCHAR(50),
    bag_of_words JSONB
);
```

Además, crear una tabla para almacenar los stopwords

```sql
CREATE TABLE stopwords (
    id SERIAL PRIMARY KEY,
    word TEXT UNIQUE NOT NULL
);
```

### Carga de datos en PostgreSQL

Proceder a cargar el dataset de noticias `news_es.csv` y el dataset de stopwords `stoplist_es.txt`.

### Leer desde PostgreSQL con Python

Completar la función para conectarte a PostgreSQL y leer los datos:

In [1]:
import psycopg2
import pandas as pd

def connect_db():
    conn = psycopg2.connect(
        dbname="lab8",    #CAMBIAR AQUI 
        user="postgres",   #CAMBIAR AQUI 
        password="postgres",   #CAMBIAR AQUI 
        host="localhost",
        port="5432"
    )
    return conn

def fetch_data():
    conn = connect_db()
    query = "SELECT id, contenido FROM noticias;"
    df = pd.read_sql(query, conn)
    conn.close()
    return df

noticias_df = fetch_data()


C:\Users\Paris Herrera\AppData\Local\Temp\ipykernel_25752\1484631387.py:17: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


## 2. (4 puntos) Preprocesamiento de texto

Implementar la función `preprocess` que reciba un texto y realice los siguiente:
- Convertir el texto a minuscula.
- Tokenización.
- Eliminación de stopwords
- Stemming (raíz de las palabras)

In [2]:
import re
import psycopg2
import pandas as pd
from nltk.stem import SnowballStemmer


def fetch_stopwords():
    conn = connect_db()
    
    query = """
        SELECT word
        FROM stopwords;
    """
    
    df = pd.read_sql(query, conn)
    conn.close()
    
    stop_words = set(
        df["word"]
        .dropna()
        .astype(str)
        .str.lower()
        .str.strip()
    )
    
    return stop_words


#Cargamos las stopwords
stop_words = fetch_stopwords()

#Stemmer para español
stemmer = SnowballStemmer("spanish")


def preprocess(text):
    #Porseacaso
    text = str(text)
    
    text = text.lower()
    
    #Tokenizacion
    tokens = re.findall(r'\b[a-záéíóúñü]+\b', text)
    
    #Eliminar stopwords
    tokens = [token for token in tokens if token not in stop_words]
    
    #Aplicar stemming
    tokens = [stemmer.stem(token) for token in tokens]
    
    return tokens


preprocess("¡Hola! Esto es una prueba, con números 123 y símbolos #@$%. ¿Funcionará bien?")

C:\Users\Paris Herrera\AppData\Local\Temp\ipykernel_25752\1214285838.py:15: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


['hol', 'prueb', 'numer', 'simbol', 'funcion']

Luego, implementar una función para calcular la frecuencia de términos:

In [3]:
from collections import Counter

def compute_bow(text):
    tokens = preprocess(text)
    
    #Calcular la frecuencia de cada token
    bow = Counter(tokens)
    
    return dict(bow)

compute_bow("Esta es una prueba con prueba y más prueba. ¡Prueba!. Programadores programadores programando.")

{'prueb': 4, 'program': 3}

## 3. (3 puntos) Actualizar la base de datos con los Bag of Words

Guardar el resultado del Bag of Words en la columna `bag_of_words` de la tabla:

In [4]:
from psycopg2.extras import Json


def update_bow_in_db(dataframe):
    conn = connect_db()
    cursor = conn.cursor()
    
    try:
        for _, row in dataframe.iterrows():
            noticia_id = row["id"]
            contenido = row["contenido"]
            
            #Se calcula el bag of words de cada contenido 
            bow = compute_bow(contenido)
            
            #Actualizamos la columna bag_of_words
            query = """
                UPDATE noticias
                SET bag_of_words = %s
                WHERE id = %s;
            """
            
            cursor.execute(query, (Json(bow), noticia_id))
        
        #Confirmamos actualizacion 
        conn.commit()
        print("Base de datos actualizada correctamente con los Bag of Words.")
    
    except Exception as e:
        conn.rollback()
        print("Error al actualizar la base de datos:", e)
    
    finally:
        cursor.close()
        conn.close()


update_bow_in_db(noticias_df)

Base de datos actualizada correctamente con los Bag of Words.


In [5]:
def fetch_data_bow():
    conn = connect_db()
    
    query = """
        SELECT id, contenido, bag_of_words
        FROM noticias
        ORDER BY id;
    """
    
    df = pd.read_sql(query, conn)
    conn.close()
    
    return df

noticias_bow_df = fetch_data_bow()
noticias_bow_df.head()

C:\Users\Paris Herrera\AppData\Local\Temp\ipykernel_25752\2257630020.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


,id,contenido,bag_of_words
0,8,Durante el foro La banca articulador empresari...,"{'aca': 1, 'adn': 1, 'cas': 1, 'for': 1, 'ret'..."
1,9,El regulador de valores de China dijo el domin...,"{'dat': 1, 'deb': 1, 'inc': 1, 'med': 1, 'mes'..."
2,10,En una industria históricamente masculina como...,"{'go': 1, 'air': 1, 'baj': 1, 'ceo': 2, 'hij':..."
3,11,Con el dato de marzo el IPC interanual encaden...,"{'agu': 1, 'car': 2, 'cas': 1, 'dat': 2, 'ine'..."
4,12,Ayer en Cartagena se dio inicio a la versión n...,"{'agu': 1, 'cup': 2, 'deb': 2, 'etc': 1, 'for'..."


## 4. (8 puntos) Consulta booleana con filtrado por keywords

Antes de aplicar el filtrado desde Python, es importante entender cómo funciona la consulta de una clave dentro de una columna JSONB en PostgreSQL. 

### Ejemplo consulta SQL en JSON:

```sql
SELECT * FROM noticias WHERE bag_of_words ? 'keyword';
```

Esta consulta selecciona todos los registros en los que el `bag_of_words` (formato JSONB) contiene una clave igual a `'keyword'`. El operador `?` verifica la existencia de una clave dentro de un JSON.

### Consulta booleana 
Implementar una función que permita parsear una consulta textual con conectores AND, OR y AND-NOT y con ello se aplique el filtro correspondiente directamente desde la base de datos. 


In [6]:
def build_keyword_condition(keyword):
    tokens = preprocess(keyword)

    if len(tokens) == 0:
        return "FALSE", []

    conditions = []
    params = []

    for token in tokens:
        conditions.append("bag_of_words ? %s")
        params.append(token)

    condition_sql = "(" + " AND ".join(conditions) + ")"

    return condition_sql, params


def apply_boolean_query(query):
    query = query.strip()

    if re.search(r"\bAND\s+NOT\b", query, flags=re.IGNORECASE):
        parts = re.split(r"\bAND\s+NOT\b", query, maxsplit=1, flags=re.IGNORECASE)

        left_keyword = parts[0].strip()
        right_keyword = parts[1].strip()

        left_condition, left_params = build_keyword_condition(left_keyword)
        right_condition, right_params = build_keyword_condition(right_keyword)

        where_clause = left_condition + " AND NOT " + right_condition
        params = left_params + right_params

    elif re.search(r"\bAND\b", query, flags=re.IGNORECASE):
        parts = re.split(r"\bAND\b", query, maxsplit=1, flags=re.IGNORECASE)

        left_keyword = parts[0].strip()
        right_keyword = parts[1].strip()

        left_condition, left_params = build_keyword_condition(left_keyword)
        right_condition, right_params = build_keyword_condition(right_keyword)

        where_clause = left_condition + " AND " + right_condition
        params = left_params + right_params

    elif re.search(r"\bOR\b", query, flags=re.IGNORECASE):
        parts = re.split(r"\bOR\b", query, maxsplit=1, flags=re.IGNORECASE)

        left_keyword = parts[0].strip()
        right_keyword = parts[1].strip()

        left_condition, left_params = build_keyword_condition(left_keyword)
        right_condition, right_params = build_keyword_condition(right_keyword)

        where_clause = left_condition + " OR " + right_condition
        params = left_params + right_params

    else:
        where_clause, params = build_keyword_condition(query)

    sql = """
        SELECT id, url, contenido, categoria, bag_of_words
        FROM noticias
        WHERE """ + where_clause + """
        ORDER BY id;
    """

    conn = connect_db()
    df = pd.read_sql(sql, conn, params=params)
    conn.close()

    return df

### Pruebas funcionales

Realizar al menos 8 pruebas funcionales con mas de dos keywords de consulta:

In [7]:
import warnings

warnings.filterwarnings(
    "ignore",
    message="pandas only supports SQLAlchemy connectable.*"
)

test_queries = [
    "transformación AND sostenible",
    "México OR Perú",
    "México AND NOT Perú",
    "banco AND digital",
    "educación OR universidad",
    "tecnología AND innovación",
    "empresa AND NOT gobierno",
    "energía OR petróleo",
    "cliente AND solución",
    "Colombia AND economía"
]

for query in test_queries:
    print(f"Probando consulta: '{query}'")
    
    results = apply_boolean_query(query)
    
    if results.empty:
        print("No se encontraron documentos.")
    else:
        print("Resultados encontrados:")
        print(results[["id", "contenido"]].head())
    
    print("-" * 60)

Probando consulta: 'transformación AND sostenible'
Resultados encontrados:
   id                                          contenido
0  12  Ayer en Cartagena se dio inicio a la versión n...
1  33  La responsable de soluciones para clientes y b...
2  37  El presidente de BBVA, Carlos Torres Vila, ha ...
3  59  La importancia que cada día más empresas le ot...
4  68  LSQA concreta el lanzamiento de la 'certificac...
------------------------------------------------------------
Probando consulta: 'México OR Perú'
Resultados encontrados:
   id                                          contenido
0  14  El país tiene pendiente generar más y mejores ...
1  19  BBVA ha presentado BBVA Spark, su propuesta in...
2  28  En las últimas décadas las empresas globales h...
3  36  México vive un momento económico en el que des...
4  42  Colombia siempre ha vivido de espaldas al Océa...
------------------------------------------------------------
Probando consulta: 'México AND NOT Perú'
Resultados encontr

## 5. (3 puntos) Actividad Final
- Medir el tiempo de ejecución de las consultas con diferentes tamaños de datos y optimizar el código según sea necesario.


**Entregable:** informe de los resultados obtenidos en formato PDF.

EJECUCIÓN DE LAS 8 CONSULTAS DEL PUNTO ANTERIOR - SIN INDICES

In [8]:
import time
import pandas as pd
import warnings

warnings.filterwarnings(
    "ignore",
    message="pandas only supports SQLAlchemy connectable.*"
)

# Consultas del punto 4 
consultas = [
    ("transformación", "AND", "sostenible"),
    ("México", "OR", "Perú"),
    ("México", "AND NOT", "Perú"),
    ("banco", "AND", "digital"),
    ("educación", "OR", "universidad"),
    ("tecnología", "AND", "innovación"),
    ("empresa", "AND NOT", "gobierno"),
    ("energía", "OR", "petróleo"),
    ("cliente", "AND", "solución"),
    ("Colombia", "AND", "economía")
]

tamanios = [300, 600, 1217]

# Eliminar índice si existe 
conn = connect_db()
cursor = conn.cursor()

cursor.execute("""
    DROP INDEX IF EXISTS idx_noticias_bag_of_words_gin;
""")

cursor.execute("""
    ANALYZE noticias;
""")

conn.commit()
cursor.close()
conn.close()

print("Pruebas SIN índice GIN")
print("=" * 70)

resultados_sin_indice = []

for tamanio in tamanios:
    print(f"\nTamaño de datos: {tamanio} registros")
    print("-" * 70)
    
    tiempos_tamanio = []
    
    for keyword_1, operador, keyword_2 in consultas:
        tokens_1 = preprocess(keyword_1)
        tokens_2 = preprocess(keyword_2)
        
        condicion_1 = " AND ".join(["bag_of_words ? %s"] * len(tokens_1))
        condicion_2 = " AND ".join(["bag_of_words ? %s"] * len(tokens_2))
        
        if condicion_1 == "":
            condicion_1 = "FALSE"
        
        if condicion_2 == "":
            condicion_2 = "FALSE"
        
        if operador == "AND":
            condicion_sql = f"({condicion_1}) AND ({condicion_2})"
        elif operador == "OR":
            condicion_sql = f"({condicion_1}) OR ({condicion_2})"
        elif operador == "AND NOT":
            condicion_sql = f"({condicion_1}) AND NOT ({condicion_2})"
        
        sql = f"""
            SELECT id, contenido
            FROM noticias
            WHERE id <= %s
            AND {condicion_sql}
            ORDER BY id;
        """
        
        params = [tamanio] + tokens_1 + tokens_2
        
        inicio = time.perf_counter()
        
        conn = connect_db()
        df_resultado = pd.read_sql(sql, conn, params=params)
        conn.close()
        
        fin = time.perf_counter()
        
        tiempo = fin - inicio
        tiempos_tamanio.append(tiempo)
        
        consulta_texto = f"{keyword_1} {operador} {keyword_2}"
        
        resultados_sin_indice.append({
            "fase": "Sin índice",
            "tamaño_datos": tamanio,
            "consulta": consulta_texto,
            "tiempo_segundos": tiempo,
            "cantidad_resultados": len(df_resultado)
        })
        
        print(f"Consulta: {consulta_texto}")
        print(f"Tiempo: {tiempo:.6f} segundos")
        print(f"Resultados encontrados: {len(df_resultado)}")
        print()
    
    promedio = sum(tiempos_tamanio) / len(tiempos_tamanio)
    print(f"PROMEDIO para {tamanio} registros SIN índice: {promedio:.6f} segundos")
    print("=" * 70)

df_sin_indice = pd.DataFrame(resultados_sin_indice)
df_sin_indice

Pruebas SIN índice GIN

Tamaño de datos: 300 registros
----------------------------------------------------------------------
Consulta: transformación AND sostenible
Tiempo: 0.037242 segundos
Resultados encontrados: 20

Consulta: México OR Perú
Tiempo: 0.043337 segundos
Resultados encontrados: 162

Consulta: México AND NOT Perú
Tiempo: 0.139938 segundos
Resultados encontrados: 23

Consulta: banco AND digital
Tiempo: 0.037684 segundos
Resultados encontrados: 44

Consulta: educación OR universidad
Tiempo: 0.040131 segundos
Resultados encontrados: 127

Consulta: tecnología AND innovación
Tiempo: 0.036556 segundos
Resultados encontrados: 40

Consulta: empresa AND NOT gobierno
Tiempo: 0.039372 segundos
Resultados encontrados: 94

Consulta: energía OR petróleo
Tiempo: 0.046842 segundos
Resultados encontrados: 123

Consulta: cliente AND solución
Tiempo: 0.047992 segundos
Resultados encontrados: 34

Consulta: Colombia AND economía
Tiempo: 0.039538 segundos
Resultados encontrados: 44

PROMEDIO 

,fase,tamaño_datos,consulta,tiempo_segundos,cantidad_resultados
0,Sin índice,300,transformación AND sostenible,0.037242,20
1,Sin índice,300,México OR Perú,0.043337,162
2,Sin índice,300,México AND NOT Perú,0.139938,23
3,Sin índice,300,banco AND digital,0.037684,44
4,Sin índice,300,educación OR universidad,0.040131,127
5,Sin índice,300,tecnología AND innovación,0.036556,40
6,Sin índice,300,empresa AND NOT gobierno,0.039372,94
7,Sin índice,300,energía OR petróleo,0.046842,123
8,Sin índice,300,cliente AND solución,0.047992,34
9,Sin índice,300,Colombia AND economía,0.039538,44


EJECUCIÓN DE LAS 8 CONSULTAS DEL PUNTO ANTERIOR - CON ÍNDICE


In [9]:
#Se crear índice GIN sobre la columna bag_of_words
conn = connect_db()
cursor = conn.cursor()

cursor.execute("""
    CREATE INDEX IF NOT EXISTS idx_noticias_bag_of_words_gin
    ON noticias
    USING GIN (bag_of_words);
""")

cursor.execute("""
    ANALYZE noticias;
""")

conn.commit()
cursor.close()
conn.close()

print("Pruebas CON índice GIN")
print("=" * 70)

resultados_con_indice = []

for tamanio in tamanios:
    print(f"\nTamaño de datos: {tamanio} registros")
    print("-" * 70)
    
    tiempos_tamanio = []
    
    for keyword_1, operador, keyword_2 in consultas:
        tokens_1 = preprocess(keyword_1)
        tokens_2 = preprocess(keyword_2)
        
        condicion_1 = " AND ".join(["bag_of_words ? %s"] * len(tokens_1))
        condicion_2 = " AND ".join(["bag_of_words ? %s"] * len(tokens_2))
        
        if condicion_1 == "":
            condicion_1 = "FALSE"
        
        if condicion_2 == "":
            condicion_2 = "FALSE"
        
        if operador == "AND":
            condicion_sql = f"({condicion_1}) AND ({condicion_2})"
        elif operador == "OR":
            condicion_sql = f"({condicion_1}) OR ({condicion_2})"
        elif operador == "AND NOT":
            condicion_sql = f"({condicion_1}) AND NOT ({condicion_2})"
        
        sql = f"""
            SELECT id, contenido
            FROM noticias
            WHERE id <= %s
            AND {condicion_sql}
            ORDER BY id;
        """
        
        params = [tamanio] + tokens_1 + tokens_2
        
        inicio = time.perf_counter()
        
        conn = connect_db()
        df_resultado = pd.read_sql(sql, conn, params=params)
        conn.close()
        
        fin = time.perf_counter()
        
        tiempo = fin - inicio
        tiempos_tamanio.append(tiempo)
        
        consulta_texto = f"{keyword_1} {operador} {keyword_2}"
        
        resultados_con_indice.append({
            "fase": "Con índice GIN",
            "tamaño_datos": tamanio,
            "consulta": consulta_texto,
            "tiempo_segundos": tiempo,
            "cantidad_resultados": len(df_resultado)
        })
        
        print(f"Consulta: {consulta_texto}")
        print(f"Tiempo: {tiempo:.6f} segundos")
        print(f"Resultados encontrados: {len(df_resultado)}")
        print()
    
    promedio = sum(tiempos_tamanio) / len(tiempos_tamanio)
    print(f"PROMEDIO para {tamanio} registros CON índice GIN: {promedio:.6f} segundos")
    print("=" * 70)

df_con_indice = pd.DataFrame(resultados_con_indice)
df_con_indice

Pruebas CON índice GIN

Tamaño de datos: 300 registros
----------------------------------------------------------------------
Consulta: transformación AND sostenible
Tiempo: 0.034373 segundos
Resultados encontrados: 20

Consulta: México OR Perú
Tiempo: 0.037592 segundos
Resultados encontrados: 162

Consulta: México AND NOT Perú
Tiempo: 0.034556 segundos
Resultados encontrados: 23

Consulta: banco AND digital
Tiempo: 0.037997 segundos
Resultados encontrados: 44

Consulta: educación OR universidad
Tiempo: 0.035769 segundos
Resultados encontrados: 127

Consulta: tecnología AND innovación
Tiempo: 0.034294 segundos
Resultados encontrados: 40

Consulta: empresa AND NOT gobierno
Tiempo: 0.036746 segundos
Resultados encontrados: 94

Consulta: energía OR petróleo
Tiempo: 0.035371 segundos
Resultados encontrados: 123

Consulta: cliente AND solución
Tiempo: 0.034876 segundos
Resultados encontrados: 34

Consulta: Colombia AND economía
Tiempo: 0.035187 segundos
Resultados encontrados: 44

PROMEDIO 

,fase,tamaño_datos,consulta,tiempo_segundos,cantidad_resultados
0,Con índice GIN,300,transformación AND sostenible,0.034373,20
1,Con índice GIN,300,México OR Perú,0.037592,162
2,Con índice GIN,300,México AND NOT Perú,0.034556,23
3,Con índice GIN,300,banco AND digital,0.037997,44
4,Con índice GIN,300,educación OR universidad,0.035769,127
5,Con índice GIN,300,tecnología AND innovación,0.034294,40
6,Con índice GIN,300,empresa AND NOT gobierno,0.036746,94
7,Con índice GIN,300,energía OR petróleo,0.035371,123
8,Con índice GIN,300,cliente AND solución,0.034876,34
9,Con índice GIN,300,Colombia AND economía,0.035187,44
